<a href="https://colab.research.google.com/github/U-mar-404/AIPDD-Lab06/blob/main/Lab_06.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 06 Lab – University Academic Helpdesk (Semantic FAQ Retrieval + Gradio)
**Name:** Muhammadn Umar  **Roll No:** 231243

Flow: Phase A (retrieval) → B (Gradio Interface) → C (validation + threshold) → D (Blocks) → E (testing).

## Setup

In [1]:
!pip install -q gradio sentence-transformers numpy

## Phase A – Semantic Retrieval

In [2]:
from sentence_transformers import SentenceTransformer, util

FAQS = [
    ("How do I register my final year project?", "Contact the FYP office and follow the published registration procedure."),
    ("How can I withdraw from a course?", "Consult the academic office about course withdrawal deadlines and approval."),
    ("How can I reserve laboratory equipment?", "Submit a lab equipment booking request to the laboratory staff."),
    ("Where can I find internship application guidance?", "Consult the career office and internship notices."),
    ("How can I request my academic transcript?", "Apply through the registrar's published transcript procedure."),
    ("How can I reset my student portal password?", "Use the official portal recovery process or contact IT support."),
]

model = SentenceTransformer("all-MiniLM-L6-v2")                 # loaded ONCE
faq_questions = [q for q, a in FAQS]
faq_vectors = model.encode(faq_questions, convert_to_tensor=True)  # encoded ONCE

def retrieve(question):
    query_vector = model.encode(question, convert_to_tensor=True)
    sims = util.cos_sim(query_vector, faq_vectors)[0]
    idx = int(sims.argmax().item())
    return FAQS[idx][0], FAQS[idx][1], float(sims[idx].item())

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [3]:
# Tasks A1 & A2: paraphrase tests
for q in ["How can I enrol my final year project?",
          "I want to quit a subject",
          "How do I get my transcript?",
          "I forgot my portal password"]:
    m, a, s = retrieve(q)
    print(f"{q!r}\n  -> {m}  (sim={s:.3f})\n")

'How can I enrol my final year project?'
  -> How do I register my final year project?  (sim=0.804)

'I want to quit a subject'
  -> How can I withdraw from a course?  (sim=0.575)

'How do I get my transcript?'
  -> How can I request my academic transcript?  (sim=0.870)

'I forgot my portal password'
  -> How can I reset my student portal password?  (sim=0.744)



**A3 – Why precompute FAQ embeddings?**
The FAQ list is fixed, so its embeddings never change. Encoding them once means each query only needs **one** model call (for the query) plus a fast cosine comparison. Recomputing all FAQ embeddings per query would waste time and CPU.

## Phase B + C – Gradio Interface with validation, threshold and status

In [4]:
import gradio as gr

THRESHOLD = 0.40   # trial value – revise after testing (see Phase C1)

def answer_question(question):
    # Input validation (no inference on empty input)
    if question is None or not str(question).strip():
        return "-", "Please type a question first.", 0.0, "Invalid input"
    try:
        matched, answer, sim = retrieve(question.strip())
    except Exception as e:
        return "-", f"Error: {e}", 0.0, "Error"
    # Out-of-scope fallback
    if sim < THRESHOLD:
        return "-", "No sufficiently relevant FAQ found. Please rephrase your question.", round(sim, 3), "Needs clarification"
    return matched, answer, round(sim, 3), "Matched"

examples = [
    "How do I register my FYP?",
    "I want to drop a subject",
    "How can I book lab equipment?",
    "I forgot my portal password",
    "Tell me the cricket score",
]

demo = gr.Interface(
    fn=answer_question,
    inputs=gr.Textbox(label="Your academic question", lines=2,
                      placeholder="e.g. How do I get my transcript?"),
    outputs=[gr.Textbox(label="Matched FAQ"),
             gr.Textbox(label="Suggested answer"),
             gr.Number(label="Cosine similarity (NOT a probability)"),
             gr.Textbox(label="Status")],
    examples=examples,
    title="University Academic Helpdesk",
    description="Type a question in everyday language. Answers are illustrative lab content, not official university policy.",
)
demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://8e139f814cd1106bdb.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


**B4 – Mapping of return values to outputs:** Gradio maps the returned values **by position**:
1st value → *Matched FAQ* Textbox, 2nd → *Suggested answer* Textbox, 3rd → *Cosine similarity* Number, 4th → *Status* Textbox.

In [5]:
demo.close()   # stop the first app before building the Blocks version

Closing server running on port: 7860


### Task C1 – Compare thresholds

In [6]:
relevant  = ["How do I register my FYP?", "I want to drop a subject", "How can I book lab equipment?"]
unrelated = ["Tell me the cricket score", "What is the weather today?", "asdfgh jkl"]

for name, qs in [("RELEVANT", relevant), ("UNRELATED", unrelated)]:
    print(name)
    for q in qs:
        _, _, s = retrieve(q)
        print(f"  {q:<35} sim={s:.3f}   T=0.30:{'accept' if s>=0.30 else 'reject'}   T=0.50:{'accept' if s>=0.50 else 'reject'}")

RELEVANT
  How do I register my FYP?           sim=0.484   T=0.30:accept   T=0.50:reject
  I want to drop a subject            sim=0.391   T=0.30:accept   T=0.50:reject
  How can I book lab equipment?       sim=0.722   T=0.30:accept   T=0.50:accept
UNRELATED
  Tell me the cricket score           sim=0.101   T=0.30:reject   T=0.50:reject
  What is the weather today?          sim=-0.035   T=0.30:reject   T=0.50:reject
  asdfgh jkl                          sim=0.042   T=0.30:reject   T=0.50:reject


**C1 – Observation (fill in after running):** Threshold 0.30 accepts ____, threshold 0.50 rejects ____. I chose THRESHOLD = ____ because ____.

**C2 – Borderline/false match (fill in):** Query: "____" → matched "____" with similarity ____ (should have been ____).

**C3 – Why the top result may still be wrong:** The highest cosine similarity only means "closest among the stored FAQs", not "correct". An out-of-scope or ambiguous question still has *some* closest FAQ, shared words/topics can inflate similarity, and the FAQ set is tiny. Cosine similarity is not a calibrated probability.

## Phase D – Blocks with two tabs and events

In [7]:
def browse(topic):
    # No AI model here – just a dictionary lookup
    return dict(FAQS).get(topic, "Select a topic.")

with gr.Blocks() as demo2:
    gr.Markdown("# University Academic Helpdesk\nAnswers are illustrative, not official policy.")

    with gr.Tab("Ask a Question"):
        question = gr.Textbox(label="Question", placeholder="Type your question...")
        search_btn = gr.Button("Search")
        matched = gr.Textbox(label="Matched FAQ")
        answer = gr.Textbox(label="Suggested answer")
        similarity = gr.Number(label="Similarity")
        status = gr.Textbox(label="Status")
        search_btn.click(answer_question, inputs=question,
                         outputs=[matched, answer, similarity, status])   # D1
        gr.Examples(examples, inputs=question)

    with gr.Tab("Browse FAQs"):
        dd = gr.Dropdown(choices=faq_questions, label="Select an FAQ topic")
        shown = gr.Textbox(label="Answer")
        dd.change(browse, inputs=dd, outputs=shown)                       # D2

demo2.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://e75bfa82c26aef3526.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


**D3 – Why two tabs ≠ two AI models:** Tabs are only a *GUI layout*. The "Ask" tab uses one embedding model; the "Browse" tab is a plain dictionary lookup with no AI. Number of AI models depends on inference code, not on the number of interface tabs.

## Phase E – Test matrix (actual outputs)

In [8]:
tests = [
    ("How do I register my FYP?", "FYP answer"),
    ("I want to drop a subject", "Course withdrawal"),
    ("How can I book lab equipment?", "Lab booking"),
    ("", "Validation message"),
    ("Tell me the cricket score", "Fallback"),
    ("asdfgh jkl", "Graceful handling"),
    ("Where do I apply for an internship?", "Internship FAQ"),      # my own test 1
    ("How do I get my official transcript?", "Transcript FAQ"),     # my own test 2
]
print(f"{'Input':<38}{'Expected':<20}{'Status':<20}{'Sim':<7}Matched")
for q, exp in tests:
    m, a, s, st = answer_question(q)
    print(f"{q!r:<38}{exp:<20}{st:<20}{s:<7}{m}")

Input                                 Expected            Status              Sim    Matched
'How do I register my FYP?'           FYP answer          Matched             0.484  How do I register my final year project?
'I want to drop a subject'            Course withdrawal   Needs clarification 0.391  -
'How can I book lab equipment?'       Lab booking         Matched             0.722  How can I reserve laboratory equipment?
''                                    Validation message  Invalid input       0.0    -
'Tell me the cricket score'           Fallback            Needs clarification 0.101  -
'asdfgh jkl'                          Graceful handling   Needs clarification 0.042  -
'Where do I apply for an internship?' Internship FAQ      Matched             0.791  Where can I find internship application guidance?
'How do I get my official transcript?'Transcript FAQ      Matched             0.859  How can I request my academic transcript?


Copy the printed results into your test table and mark **Pass/Fail honestly**. If an out-of-scope question (e.g. cricket) is *not* rejected, record it as a limitation – improvement: raise the threshold, add an "out-of-scope" example FAQ, or use a margin between the top-1 and top-2 scores.

## Reflection (150–250 words – write in your own words)
Mention: (1) the GUI (Gradio) is separate from the AI part (`retrieve`); (2) the model is loaded once outside the handler for speed; (3) limits: tiny FAQ set, similarity ≠ probability, threshold is a guess; (4) improvements: top-3 suggestions, larger FAQ set, better threshold tuning, user feedback button.

## Viva quick answers
- **GUI / AI / logic lines:** GUI = `gr.Interface` / `gr.Blocks`; AI = `model.encode` + `util.cos_sim` in `retrieve`; logic = validation + threshold in `answer_question`.
- **Why model outside handler:** loading is slow; doing it per request would make every query slow.
- **Data types:** receives `str`, returns `(str, str, float, str)`.
- **Interface vs Blocks:** Interface = quick fixed layout (inputs → function → outputs); Blocks = custom layout, tabs and events like `.click()` / `.change()`.
- **Highest similarity ≠ correct:** see C3.
- **Adapt to image/ASR:** keep the same GUI, replace the input with `gr.Image` / `gr.Audio` and the function with a classifier / ASR model.